# YOLO26n on VOC — V2: Classification-Head Transplant + Fine-Tuning

**Continues from V1** (`yolo25n_voc_v1_baseline.ipynb`), which produced:

- A working Pascal VOC dataset in YOLO format.
- A VOC-idx -> COCO-idx `index_map` (by shared class meaning).
- A zero-shot baseline report (`reports/baseline_*.json`) for the untouched
  COCO-pretrained model, evaluated both in its native 80-class space and
  restricted to the 20 VOC-relevant classes.

**This notebook (V2):**

1. Load `yolo26n.pt` and inspect the actual `Detect` head structure (don't
   assume anything — this model's head turns out to be an end-to-end,
   NMS-free design with duplicate one2many/one2one branches).
2. Build a 20-channel classification layer per scale **and per branch**, by
   copying only the weight rows/biases for the 20 VOC-relevant COCO classes
   (via V1's `index_map`), in VOC index order.
3. Leave everything else — backbone, neck, box-regression branch, and even
   the earlier (non-class-indexed) layers of the classification branch —
   untouched from the pretrained checkpoint.
4. Validate the transplanted model natively on VOC (nc=20) *before* any
   fine-tuning, as a sanity check and as a genuine "transplant-only" data
   point for the final comparison.
5. Fine-tune from that transplanted starting point — using a custom trainer
   that bypasses a real Ultralytics gotcha (see Section 5 markdown below).
6. Validate the fine-tuned model natively on VOC's val split.
7. Load the V1 baseline report and build an overall + per-class delta
   comparator across three points: zero-shot baseline -> transplant-only
   (0 fine-tuning steps) -> fine-tuned.
8. Persist fine-tuned weights, the training run, and the comparison report.

**Design note carried over from V1:** dataset stays local/ephemeral to the
Colab session; only outputs (weights, reports) are meant to be saved off.

## 1. Environment Setup

In [ ]:
import os

PROJECT_ROOT = '/content/yolo25n_voc'
for sub in ['runs', 'configs', 'reports', 'weights']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)

print(f'Project root (local, ephemeral): {PROJECT_ROOT}')

# ── Install Ultralytics ──
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

import torch
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > select a GPU, then re-run."
)
print(f"Device: {torch.cuda.get_device_name(0)}")
print(f"VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

### 1a. Bring in the V1 baseline report

V1's outputs were **not** persisted to Drive (kept local + downloaded to your
machine). Upload that `baseline_*.json` file here so V2 can diff against it
in Section 7.

In [ ]:
from google.colab import files

print("Upload the V1 baseline_*.json report:")
uploaded = files.upload()
baseline_report_path = next(iter(uploaded.keys()))
print(f"Using baseline report: {baseline_report_path}")

## 2. Dataset — VOC, native 20-class format

Unlike V1 (which needed VOC's val labels remapped into COCO's 80-class index
space to evaluate the untouched COCO head), V2 fine-tunes and validates
**natively** in VOC's own 20-class space — no remapping needed here. We still
use Ultralytics' built-in `VOC.yaml`, so class *order* is guaranteed to be
byte-for-byte identical to what V1 used to build `index_map`.

In [ ]:
from ultralytics.data.utils import check_det_dataset

voc_data = check_det_dataset('VOC.yaml')
voc_names = voc_data['names']  # {idx: name}, 20 classes — same order as V1
print(f"Classes (nc={voc_data['nc']}): {voc_names}")

## 3. Model — Inspect the actual `Detect` head (don't assume shapes)

yolo26n turns out to use an **end-to-end (NMS-free) head**: alongside the
usual `cv2`/`cv3` (one2many) branches used during training, it has a
duplicate `one2one_cv2`/`one2one_cv3` pair used for NMS-free inference. Any
class-head transplant that only touches `cv3` would leave `one2one_cv3`
still speaking COCO's 80 classes — silently breaking NMS-free
inference/export while looking fine in training-mode `val()`. We confirm
this structure by inspection rather than assuming it.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo26n.pt')
detect = model.model.model[-1]

print(f"Detect class:        {type(detect).__name__}")
print(f"end2end:              {getattr(detect, 'end2end', False)}")
print(f"nc (native):           {detect.nc}")
print(f"nl (scale levels):     {detect.nl}")
print(f"reg_max:               {detect.reg_max}   ->  no = nc + 4*reg_max = {detect.no}")
print(f"has cv3 (one2many cls branch):        {hasattr(detect, 'cv3')}")
print(f"has one2one_cv3 (one2one cls branch): {hasattr(detect, 'one2one_cv3')}")

print("\nPer-level classification-branch final layer (the ONLY class-indexed layer):")
for i, level in enumerate(detect.cv3):
    final = level[-1]
    print(f"  level {i}: {type(final).__name__}  weight={tuple(final.weight.shape)}  "
          f"bias={tuple(final.bias.shape) if final.bias is not None else None}")

assert hasattr(detect, "cv3") and isinstance(detect.cv3[0][-1], torch.nn.Conv2d), (
    "Architecture assumption violated: expected Detect.cv3[i][-1] to be a plain nn.Conv2d "
    "classification layer. Re-inspect before proceeding — do not blindly transplant."
)

## 4. Class Alignment (same as V1) — build `index_map`

Re-derive the same VOC-idx -> COCO-idx mapping V1 used, from the same
correspondence table, against this model's own `names` dict (rather than
trusting it stayed identical across notebooks).

In [ ]:
VOC_TO_COCO_NAME = {
    'aeroplane':   'airplane',
    'bicycle':     'bicycle',
    'bird':        'bird',
    'boat':        'boat',
    'bottle':      'bottle',
    'bus':         'bus',
    'car':         'car',
    'cat':         'cat',
    'chair':       'chair',
    'cow':         'cow',
    'diningtable': 'dining table',
    'dog':         'dog',
    'horse':       'horse',
    'motorbike':   'motorcycle',
    'person':      'person',
    'pottedplant': 'potted plant',
    'sheep':       'sheep',
    'sofa':        'couch',
    'train':       'train',
    'tvmonitor':   'tv',
}

def build_index_map(voc_names: dict, coco_names: dict) -> dict:
    coco_name_to_idx = {name: idx for idx, name in coco_names.items()}
    index_map = {}
    for voc_idx, voc_name in voc_names.items():
        coco_name = VOC_TO_COCO_NAME.get(voc_name)
        if coco_name is None or coco_name not in coco_name_to_idx:
            raise ValueError(f"No COCO match found for VOC class '{voc_name}' (idx {voc_idx})")
        index_map[voc_idx] = coco_name_to_idx[coco_name]
    return index_map

coco_names = model.model.names  # {idx: name}, 80 classes, BEFORE transplant
index_map = build_index_map(voc_names, coco_names)
assert sorted(index_map.keys()) == list(range(20)), "index_map must densely cover VOC idx 0..19"

print("VOC idx -> COCO idx   (VOC name -> COCO name)")
for v_idx, c_idx in index_map.items():
    print(f"  {v_idx:2d} -> {c_idx:2d}   ({voc_names[v_idx]} -> {coco_names[c_idx]})")

## 5. Classification-Head Transplant

Only the final 1x1 conv of each scale level, **in each branch** (`cv3` and
`one2one_cv3`), is class-indexed — it's the only layer whose output channels
correspond 1:1 to class identities. Everything upstream of it (the DWConv +
projection pairs that build up to that layer) operates on latent features,
not class logits, so it's left completely untouched from the pretrained
checkpoint, at its original width. We copy `weight[coco_idx]` ->
`weight[voc_idx]` (and the matching bias) for exactly the 20 mapped classes,
in VOC-index order.

In [ ]:
import torch.nn as nn

def transplant_classification_head(det_model, index_map: dict):
    """det_model: an ultralytics.nn.tasks.DetectionModel (e.g. YOLO(...).model).
    Mutates it in place: shrinks the classification branch(es) of the Detect
    head from 80 -> len(index_map) output channels, transplanting weight rows
    for mapped classes instead of randomly reinitializing them."""
    detect = det_model.model[-1]
    n_new = len(index_map)
    voc_order = [index_map[v] for v in sorted(index_map.keys())]  # coco idx, in VOC-idx order

    branches = [detect.cv3]
    if getattr(detect, "end2end", False) and hasattr(detect, "one2one_cv3"):
        branches.append(detect.one2one_cv3)

    old_nc = detect.nc
    for branch in branches:
        for level in branch:
            old_final = level[-1]
            assert isinstance(old_final, nn.Conv2d) and old_final.out_channels == old_nc
            new_final = nn.Conv2d(
                old_final.in_channels, n_new,
                kernel_size=old_final.kernel_size, stride=old_final.stride,
                padding=old_final.padding, bias=old_final.bias is not None,
            )
            with torch.no_grad():
                new_final.weight.copy_(old_final.weight[voc_order])
                if old_final.bias is not None:
                    new_final.bias.copy_(old_final.bias[voc_order])
            level[-1] = new_final

    detect.nc = n_new
    detect.no = n_new + detect.reg_max * 4
    det_model.nc = n_new
    if hasattr(det_model, "yaml") and isinstance(det_model.yaml, dict):
        det_model.yaml["nc"] = n_new
    return det_model


def verify_transplant(original_det_model, new_det_model, index_map: dict):
    """Every transplanted row must equal the original COCO row, exactly,
    for every level and every branch."""
    orig_detect, new_detect = original_det_model.model[-1], new_det_model.model[-1]
    pairs = [(orig_detect.cv3, new_detect.cv3)]
    if getattr(orig_detect, "end2end", False):
        pairs.append((orig_detect.one2one_cv3, new_detect.one2one_cv3))

    for orig_branch, new_branch in pairs:
        for lvl, (ol, nl_) in enumerate(zip(orig_branch, new_branch)):
            ow, nw = ol[-1].weight, nl_[-1].weight
            ob, nb = ol[-1].bias, nl_[-1].bias
            for v_idx, c_idx in index_map.items():
                assert torch.equal(nw[v_idx], ow[c_idx]), f"weight mismatch level {lvl}, voc_idx={v_idx}"
                if ob is not None:
                    assert torch.equal(nb[v_idx], ob[c_idx]), f"bias mismatch level {lvl}, voc_idx={v_idx}"
    return True


import copy
original_det_model = copy.deepcopy(model.model)  # keep a pristine copy to verify against
transplant_classification_head(model.model, index_map)
verify_transplant(original_det_model, model.model, index_map)
del original_det_model

print(f"Transplant verified. New head nc={model.model.model[-1].nc}, "
      f"no={model.model.model[-1].no}")
print(f"cv3 final layer shape:        {tuple(model.model.model[-1].cv3[0][-1].weight.shape)}")
print(f"one2one_cv3 final layer shape: {tuple(model.model.model[-1].one2one_cv3[0][-1].weight.shape)}")

### 5a. Persist the transplant as a standalone checkpoint

We save this as a normal Ultralytics-format `.pt` and reload it fresh via
`YOLO(...)`, rather than continuing to use the in-memory object. This matters
for Section 5b below: reloading from disk is what a plain
`model.train(data='VOC.yaml')` call would also do, so this is the realistic
starting point to fine-tune from.

In [ ]:
from datetime import datetime

def save_transplanted_checkpoint(det_model, names: dict, out_path: str):
    det_model.names = names
    det_model.nc = len(names)
    if hasattr(det_model, "yaml") and isinstance(det_model.yaml, dict):
        det_model.yaml["nc"] = len(names)

    ckpt = {
        "date": datetime.now().isoformat(),
        "version": ultralytics.__version__,
        "license": "AGPL-3.0",
        "docs": "https://docs.ultralytics.com",
        "epoch": -1,
        "best_fitness": None,
        "model": copy.deepcopy(det_model).half(),
        "ema": None,
        "updates": 0,
        "optimizer": None,
        "scaler": None,
        "train_args": {},
        "train_metrics": {},
        "train_results": {},
        "git": {},
    }
    torch.save(ckpt, out_path)
    return out_path


TRANSPLANT_CKPT = f"{PROJECT_ROOT}/weights/yolo26n_voc20_transplanted.pt"
save_transplanted_checkpoint(model.model, voc_names, TRANSPLANT_CKPT)
print(f"Transplanted checkpoint saved: {TRANSPLANT_CKPT}")

transplanted_model = YOLO(TRANSPLANT_CKPT)  # fresh reload, as fine-tuning will start from
assert transplanted_model.model.model[-1].nc == 20
assert transplanted_model.names[0] == voc_names[0]
print("Reloaded transplanted checkpoint OK.")

### 5b. A real Ultralytics gotcha, and why we need a custom trainer

Ultralytics' `DetectionTrainer.get_model()` does **not** simply reuse the
model you hand it. It always rebuilds the head from a yaml config sized for
the *target* dataset's `nc`, then transfers weights into that fresh
architecture by matching parameter name **and shape**
(`ultralytics.nn.modules.head.Detect`: the classification branch's
intermediate width is `c3 = max(ch[0], min(nc, 100))`, which is a function of
`nc`). For this model, `ch[0] = 64`, so `nc=80` gives `c3=80` (what our
transplant assumed and preserved) but `nc=20` gives `c3=64`. A stock
`model.train(data='VOC.yaml')` call would therefore rebuild the classification
branch at width 64, no longer matching our transplanted 80-wide layer by
shape — and Ultralytics would silently drop it and randomly reinitialize it
anyway, defeating the entire point of the transplant. This was **not**
theoretical: it reproduced exactly this way in testing before the fix below,
confirmed with a callback that inspected the layer weights the instant
training started.

Fix: a trainer subclass that skips that rebuild and reuses our
already-transplanted model as-is.

In [ ]:
from ultralytics.models.yolo.detect.train import DetectionTrainer
from ultralytics.utils import RANK


class TransplantedHeadTrainer(DetectionTrainer):
    """Bypasses the default cfg-based head reconstruction so the
    transplanted classification weights (and their original 80-wide
    bottleneck) survive into training unchanged."""

    def get_model(self, cfg=None, weights=None, verbose=True):
        det_model = weights
        assert det_model is not None, "TransplantedHeadTrainer requires a pre-built model."
        assert det_model.model[-1].nc == self.data["nc"], (
            f"Model head nc ({det_model.model[-1].nc}) != dataset nc ({self.data['nc']}) — "
            "did the transplant run before this?"
        )
        det_model.nc = self.data["nc"]
        det_model.names = self.data["names"]
        det_model.args = self.args
        if verbose and RANK in {-1, 0}:
            det_model.info()
        return det_model

## 6. Validate the transplanted model, natively, BEFORE any fine-tuning

This is both a sanity check (did the transplant actually produce a sane
model, or garbage?) and a genuine data point: it tells us how much the
transplant alone is worth, at zero fine-tuning steps, compared against V1's
zero-shot baseline (measured a different way — COCO-remapped labels against
the native 80-class head). Mathematically these should be close, since
restricting a softmax-style/BCE classifier's output to a class subset and
re-indexing it doesn't change its predictions for those classes.

In [ ]:
pretrain_val_results = transplanted_model.val(
    data='VOC.yaml',
    split='val',
    imgsz=640,
    project=f'{PROJECT_ROOT}/runs',
    name='transplant_zero_shot_native',
)

pretrain_metrics = {
    'map50': float(pretrain_val_results.box.map50),
    'map50_95': float(pretrain_val_results.box.map),
    'per_class_ap50': {voc_names[i]: float(v) for i, v in
                        zip(pretrain_val_results.box.ap_class_index, pretrain_val_results.box.ap50)},
    'per_class_ap50_95': {voc_names[i]: float(pretrain_val_results.box.maps[i]) for i in voc_names},
}

print("Transplant-only (0 fine-tuning steps), native 20-class validation:")
print(f"  mAP50    = {pretrain_metrics['map50']:.4f}")
print(f"  mAP50-95 = {pretrain_metrics['map50_95']:.4f}")

## 7. Fine-Tune from the Transplanted Starting Point

Standard `VOC.yaml`, native `nc=20` — no remapping needed. We pass our
`TransplantedHeadTrainer` explicitly so the transplanted weights (and their
80-wide bottleneck) are preserved as the actual training starting point,
rather than silently discarded per Section 5b.

Epoch count kept modest for a first fine-tuned version (MVP) — bump this up
once the pipeline is confirmed to behave as expected end-to-end.

In [ ]:
EPOCHS = 50   # adjust based on time budget; this is a starting point, not a target
IMGSZ = 640
BATCH = 64    # adjust to fit the assigned GPU's VRAM

finetune_results = transplanted_model.train(
    data='VOC.yaml',
    trainer=TransplantedHeadTrainer,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    project=f'{PROJECT_ROOT}/runs',
    name='finetune_from_transplant',
    pretrained=True,   # keep Ultralytics' own bookkeeping consistent; our trainer ignores the rebuild path anyway
)

FINETUNED_CKPT = f"{PROJECT_ROOT}/runs/finetune_from_transplant/weights/best.pt"
print(f"Fine-tuned weights: {FINETUNED_CKPT}")

## 8. Validate the Fine-Tuned Model — native VOC val split

In [ ]:
finetuned_model = YOLO(FINETUNED_CKPT)

finetuned_val_results = finetuned_model.val(
    data='VOC.yaml',
    split='val',
    imgsz=IMGSZ,
    project=f'{PROJECT_ROOT}/runs',
    name='finetuned_native_val',
)

finetuned_metrics = {
    'map50': float(finetuned_val_results.box.map50),
    'map50_95': float(finetuned_val_results.box.map),
    'per_class_ap50': {voc_names[i]: float(v) for i, v in
                        zip(finetuned_val_results.box.ap_class_index, finetuned_val_results.box.ap50)},
    'per_class_ap50_95': {voc_names[i]: float(finetuned_val_results.box.maps[i]) for i in voc_names},
}

print("Fine-tuned, native 20-class validation:")
print(f"  mAP50    = {finetuned_metrics['map50']:.4f}")
print(f"  mAP50-95 = {finetuned_metrics['map50_95']:.4f}")

## 9. Comparator — Baseline vs. Transplant-Only vs. Fine-Tuned

Loads the V1 baseline report and builds a three-way, per-class delta table:

- **baseline**: untouched COCO-pretrained model, zero-shot, COCO-remapped labels.
- **transplant_only**: head-transplanted model, zero-shot (0 fine-tuning steps), native labels.
- **finetuned**: after fine-tuning on VOC.

This also answers the question V1 deferred: did the transplanted starting
point actually help, independent of fine-tuning — i.e. is `transplant_only`
already close to (or better than) `baseline`'s shared-20-class numbers before
a single gradient step?

In [ ]:
import json

with open(baseline_report_path) as f:
    baseline_report = json.load(f)

baseline_per_class = {row['voc_class']: row for row in baseline_report['per_class']}

comparison_rows = []
for voc_idx in sorted(voc_names.keys()):
    cls = voc_names[voc_idx]
    b = baseline_per_class.get(cls, {'AP50': 0.0, 'AP50-95': 0.0})
    t_ap50 = pretrain_metrics['per_class_ap50'].get(cls, 0.0)
    t_ap5095 = pretrain_metrics['per_class_ap50_95'].get(cls, 0.0)
    f_ap50 = finetuned_metrics['per_class_ap50'].get(cls, 0.0)
    f_ap5095 = finetuned_metrics['per_class_ap50_95'].get(cls, 0.0)

    comparison_rows.append({
        'class': cls,
        'baseline_AP50': b['AP50'], 'transplant_only_AP50': t_ap50, 'finetuned_AP50': f_ap50,
        'delta_transplant_vs_baseline_AP50': t_ap50 - b['AP50'],
        'delta_finetuned_vs_baseline_AP50': f_ap50 - b['AP50'],
        'baseline_AP50-95': b['AP50-95'], 'transplant_only_AP50-95': t_ap5095, 'finetuned_AP50-95': f_ap5095,
        'delta_transplant_vs_baseline_AP50-95': t_ap5095 - b['AP50-95'],
        'delta_finetuned_vs_baseline_AP50-95': f_ap5095 - b['AP50-95'],
    })

overall_comparison = {
    'baseline_shared20_mAP50': baseline_report['shared_20class']['mAP50_shared_20'],
    'transplant_only_mAP50': pretrain_metrics['map50'],
    'finetuned_mAP50': finetuned_metrics['map50'],
    'baseline_shared20_mAP50-95': baseline_report['shared_20class']['mAP50-95_shared_20'],
    'transplant_only_mAP50-95': pretrain_metrics['map50_95'],
    'finetuned_mAP50-95': finetuned_metrics['map50_95'],
}

print("Overall comparison:")
for k, v in overall_comparison.items():
    print(f"  {k:32s} = {v:.4f}")

print("\nPer-class comparison:")
for row in comparison_rows:
    print(f"  {row['class']:15s}  "
          f"AP50: base={row['baseline_AP50']:.3f} transplant={row['transplant_only_AP50']:.3f} "
          f"finetuned={row['finetuned_AP50']:.3f}  (Δft={row['delta_finetuned_vs_baseline_AP50']:+.3f})")

## 10. Persist Outputs — weights, run, comparison report

In [ ]:
comparison_report_path = f"{PROJECT_ROOT}/reports/comparison_{datetime.now():%Y%m%d_%H%M%S}.json"
with open(comparison_report_path, 'w') as f:
    json.dump({
        'overall': overall_comparison,
        'per_class': comparison_rows,
        'index_map': index_map,
        'baseline_report_used': baseline_report_path,
        'finetune_epochs': EPOCHS,
    }, f, indent=2)

print(f"Comparison report saved: {comparison_report_path}")
print(f"Fine-tuned weights:      {FINETUNED_CKPT}")
print(f"Transplanted checkpoint: {TRANSPLANT_CKPT}")

# Download the key artifacts to your local machine
files.download(comparison_report_path)
files.download(FINETUNED_CKPT)

## V2 complete

We now have:

- A verified, architecture-inspected classification-head transplant (both
  one2many and one2one branches) instead of a random-init `nc=20` head.
- A custom trainer that avoids a real Ultralytics pitfall where the standard
  training path would have silently discarded the transplant.
- A transplant-only (0-step) checkpoint measured natively, isolating the
  transplant's own contribution from fine-tuning's.
- A fine-tuned model, validated natively on VOC.
- A three-way (baseline / transplant-only / fine-tuned) overall + per-class
  comparison report, persisted alongside the fine-tuned weights.

**Possible next bottleneck to look at (V3+, not implemented here):**
once real numbers are in, check whether the per-class deltas point at a data
problem (some VOC classes chronically underrepresented / hard) before reaching
for more training tricks — per the "data over models" principle.